# Macro-Driven Regime Switching: Dynamic Asset Allocation Strategy
### End-to-End Walk-Forward Backtesting Framework (Google Colab Reproducible)
**Course**: Financial Data Analytics (FDA)  
**Target Universe**: SPY (S&P 500 ETF) & AGG (iShares Core U.S. Aggregate Bond ETF, proxy-stitched with VBMFX pre-2003)  
**Models**:
- **MRS**: 2-State Econometric Hamilton Markov Regime-Switching Model (on SPY Returns + Realized Volatility)
- **ML**: 13-Factor Walk-Forward Expanding-Window Random Forest Classifier (predicting forward tail risk without look-ahead bias)

*This notebook is fully self-contained. You can open and run all cells sequentially in Google Colab or JupyterLab with zero errors.*


## 1. Environment Setup & Dependencies

In [ ]:
# Install required packages for Google Colab
!pip install -q yfinance pandas-datareader statsmodels scikit-learn matplotlib

import os
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
import pandas_datareader.data as web
from statsmodels.tsa.regime_switching.markov_regression import MarkovRegression
from sklearn.ensemble import RandomForestClassifier

warnings.filterwarnings('ignore')
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("All libraries imported successfully!")


## 2. Data Ingestion & Point-in-Time Preparation
- **Total Return Adjustment**: Uses `auto_adjust=True` for SPY and AGG to capture dividends and coupon compounding.
- **Proxy-Stitching**: Stitches AGG with Vanguard Total Bond Market Index (VBMFX) prior to AGG's September 2003 inception.
- **Publication Lag**: Applies `.shift(1)` to `CPI_Inflation_Index` and `Unemployment_Rate` to strictly eliminate look-ahead bias and reflect official Bureau of Labor Statistics (BLS) release schedules.


In [ ]:
def load_and_prepare_dataset(start_date="2000-01-01", end_date="2026-09-30"):
    start_dt = pd.to_datetime(start_date)
    fetch_start = (start_dt - pd.DateOffset(years=2)).strftime("%Y-%m-%d")
    
    print(f"1. Fetching market price data (SPY, AGG, VBMFX) from {fetch_start} to {end_date}...")
    mkt_raw = yf.download(["SPY", "AGG", "VBMFX"], start=fetch_start, end=end_date, auto_adjust=True)
    if isinstance(mkt_raw.columns, pd.MultiIndex):
        if 'Close' in mkt_raw.columns.levels[0]:
            mkt = mkt_raw['Close'].copy()
        elif 'Adj Close' in mkt_raw.columns.levels[0]:
            mkt = mkt_raw['Adj Close'].copy()
    else:
        mkt = mkt_raw[['SPY', 'AGG', 'VBMFX']].copy()
        
    # Proxy-stitch AGG using VBMFX before September 2003
    first_agg_idx = mkt['AGG'].first_valid_index()
    if first_agg_idx is not None and mkt.index[0] < first_agg_idx:
        ratio = mkt.loc[first_agg_idx, 'AGG'] / mkt.loc[first_agg_idx, 'VBMFX']
        mkt['AGG'] = mkt['AGG'].fillna(mkt['VBMFX'] * ratio)
        
    print("2. Fetching FRED macroeconomic series (10Y2Y spread, CPI, Fed Funds, Unemployment)...")
    series_ids = {
        "T10Y2Y": "10Y2Y_Yield_Spread",
        "CPIAUCSL": "CPI_Inflation_Index",
        "FEDFUNDS": "Fed_Funds_Rate",
        "UNRATE": "Unemployment_Rate"
    }
    try:
        macro = web.DataReader(list(series_ids.keys()), 'fred', start=fetch_start, end=end_date)
        macro = macro.rename(columns=series_ids)
    except Exception as e:
        print(f"Fallback to direct FRED CSV download: {e}")
        dfs = []
        for sid, name in series_ids.items():
            url = f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={sid}"
            t_df = pd.read_csv(url, parse_dates=['DATE'], index_col='DATE')
            t_df[sid] = pd.to_numeric(t_df[sid], errors='coerce')
            t_df = t_df.rename(columns={sid: name})
            dfs.append(t_df)
        macro = dfs[0]
        for d in dfs[1:]:
            macro = macro.join(d, how='outer')
        macro = macro[(macro.index >= fetch_start) & (macro.index <= end_date)]

    # 3. Monthly Resampling & Publication Lag (.shift(1) for CPI and UNRATE)
    mkt_monthly = mkt.resample('ME').last()
    macro_monthly = macro.resample('ME').last().ffill()
    
    # Apply 1-month lag to economic surveys
    macro_monthly['CPI_Inflation_Index'] = macro_monthly['CPI_Inflation_Index'].shift(1)
    macro_monthly['Unemployment_Rate'] = macro_monthly['Unemployment_Rate'].shift(1)
    macro_monthly = macro_monthly.ffill()
    
    dataset = mkt_monthly.join(macro_monthly, how='inner')
    
    # 4. Feature Engineering
    dataset['CPI_YoY'] = dataset['CPI_Inflation_Index'].pct_change(12) * 100
    dataset['SPY_3M_Ret'] = dataset['SPY'].pct_change(3)
    dataset['SPY_6M_Ret'] = dataset['SPY'].pct_change(6)
    dataset['SPY_12M_Ret'] = dataset['SPY'].pct_change(12)
    dataset['AGG_3M_Ret'] = dataset['AGG'].pct_change(3)
    dataset['AGG_12M_Ret'] = dataset['AGG'].pct_change(12)
    
    spy_daily_ret = mkt['SPY'].pct_change()
    dataset['SPY_Vol_3M'] = (spy_daily_ret.rolling(63).std() * np.sqrt(252)).resample('ME').last()
    dataset['Yield_Spread_Change_3M'] = dataset['10Y2Y_Yield_Spread'].diff(3)
    dataset['Fed_Funds_Change_3M'] = dataset['Fed_Funds_Rate'].diff(3)
    dataset['Unemployment_Change_3M'] = dataset['Unemployment_Rate'].diff(3)
    
    # 5. Ground Truth Target Labeling (Forward 3M return < -2% or Drawdown > 10%)
    rolling_max = dataset['SPY'].cummax()
    drawdown = (dataset['SPY'] - rolling_max) / rolling_max
    dataset['Drawdown'] = drawdown
    forward_3m_ret = dataset['SPY'].pct_change(3).shift(-3)
    dataset['Regime_Target'] = np.where((forward_3m_ret < -0.02) | (drawdown < -0.10), 1, 0)
    
    dataset = dataset[dataset.index >= start_dt].ffill().bfill()
    print(f"Dataset prepared successfully! Shape: {dataset.shape}")
    return dataset

df = load_and_prepare_dataset(start_date="2000-01-01", end_date="2026-09-30")
df.tail(3)


## 3. Regime Switching Models
1. **Econometric Hamilton Markov Switching Model (MRS)**: Fits on SPY returns and realized volatility to estimate smoothed posterior regime probabilities.
2. **Machine Learning Random Forest (RF)**: Walk-forward expanding-window ensemble (120 months training startup) using all 13 macro, term structure, and momentum features.


In [ ]:
# Features for Machine Learning
features_ml = [
    'SPY_3M_Ret', 'SPY_6M_Ret', 'SPY_12M_Ret',
    'AGG_3M_Ret', 'AGG_12M_Ret', 'SPY_Vol_3M',
    '10Y2Y_Yield_Spread', 'CPI_YoY', 'Fed_Funds_Rate',
    'Unemployment_Rate', 'Yield_Spread_Change_3M',
    'Fed_Funds_Change_3M', 'Unemployment_Change_3M'
]

# 1. Fit Markov Regime Switching Model
print("Fitting Markov Regime Switching Model...")
mrs_data = df[['SPY_3M_Ret', 'SPY_Vol_3M']].dropna()
mrs_model = MarkovRegression(
    endog=mrs_data['SPY_3M_Ret'],
    k_regimes=2,
    exog=mrs_data[['SPY_Vol_3M']],
    switching_variance=True
)
mrs_res = mrs_model.fit(disp=False)
regime_means = mrs_res.params.filter(like='const')
risk_off_idx = 1 if (len(regime_means) >= 2 and regime_means.iloc[0] > regime_means.iloc[1]) else 0
df['MRS_Prob'] = pd.Series(mrs_res.smoothed_marginal_probabilities[risk_off_idx], index=mrs_data.index).reindex(df.index).ffill().bfill()

# 2. Fit Walk-Forward Expanding-Window Random Forest Model
print("Running Expanding-Window Walk-Forward Random Forest...")
X = df[features_ml]
y = df['Regime_Target']
min_train_months = 120
n_samples = len(df)

ml_probs = [0.5] * min_train_months
gini_importances_list = []

for t in range(min_train_months, n_samples):
    X_train = X.iloc[:t]
    y_train = y.iloc[:t]
    X_test = X.iloc[[t]]
    
    rf = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
    rf.fit(X_train, y_train)
    ml_probs.append(rf.predict_proba(X_test)[0][1])
    gini_importances_list.append(rf.feature_importances_)

df['ML_Prob'] = pd.Series(ml_probs, index=df.index)
print("Regime probability estimation complete!")


## 4. Backtesting Engine with 10 bps Friction & Organic Drift

In [ ]:
def run_backtest_simulation(df, signal_series, allocation_type='binary', threshold=0.5, transaction_cost=0.001):
    asset_returns = df[['SPY', 'AGG']].pct_change().fillna(0)
    weights = pd.DataFrame(index=df.index, columns=['SPY', 'AGG'], dtype=float)
    
    if allocation_type == 'binary':
        weights['AGG'] = np.where(signal_series > threshold, 1.0, 0.0)
        weights['SPY'] = 1.0 - weights['AGG']
    elif allocation_type == 'continuous':
        weights['AGG'] = signal_series.clip(0, 1)
        weights['SPY'] = 1.0 - weights['AGG']
        
    weights = weights.shift(1).fillna(0.5)
    
    portfolio_returns = []
    current_weights = np.array([0.5, 0.5])
    
    for date, row in asset_returns.iterrows():
        target_weights = weights.loc[date].values
        returns = row.values
        
        ret_no_cost = np.sum(current_weights * returns)
        end_weights = current_weights * (1.0 + returns)
        sum_end = np.sum(end_weights)
        end_norm = end_weights / sum_end if sum_end > 0 else current_weights
        
        turnover = np.sum(np.abs(target_weights - end_norm))
        net_ret = ret_no_cost - turnover * transaction_cost
        portfolio_returns.append(net_ret)
        current_weights = target_weights
        
    return pd.Series(portfolio_returns, index=df.index)

def run_60_40_benchmark(df, transaction_cost=0.001):
    target = np.array([0.6, 0.4])
    asset_returns = df[['SPY', 'AGG']].pct_change().fillna(0)
    portfolio_returns = []
    current_weights = np.array([0.6, 0.4])
    
    for date, row in asset_returns.iterrows():
        returns = row.values
        ret_no_cost = np.sum(current_weights * returns)
        end_weights = current_weights * (1.0 + returns)
        sum_end = np.sum(end_weights)
        end_norm = end_weights / sum_end if sum_end > 0 else current_weights
        
        turnover = np.sum(np.abs(target - end_norm))
        net_ret = ret_no_cost - turnover * transaction_cost
        portfolio_returns.append(net_ret)
        current_weights = target
        
    return pd.Series(portfolio_returns, index=df.index)

def calculate_all_metrics(returns_series, rf=0.0):
    n_months = len(returns_series)
    cum_ret = (1 + returns_series).prod() - 1
    ann_ret = (1 + cum_ret) ** (12 / n_months) - 1
    ann_vol = returns_series.std() * np.sqrt(12)
    sharpe = (ann_ret - rf) / ann_vol if ann_vol > 0 else 0
    
    rf_m = (1 + rf) ** (1 / 12) - 1
    downside = returns_series[returns_series < rf_m] - rf_m
    downside_vol = np.sqrt(np.mean(downside**2)) * np.sqrt(12) if len(downside) > 0 else 0
    sortino = (ann_ret - rf) / downside_vol if downside_vol > 0 else 0
    
    cum_w = (1 + returns_series).cumprod()
    dd = (cum_w - cum_w.cummax()) / cum_w.cummax()
    max_dd = dd.min()
    calmar = ann_ret / abs(max_dd) if abs(max_dd) > 0 else 0
    
    return {
        "Cumulative Return": cum_ret,
        "Annualized Return": ann_ret,
        "Annualized Vol": ann_vol,
        "Sharpe Ratio": sharpe,
        "Sortino Ratio": sortino,
        "Max Drawdown": max_dd,
        "Calmar Ratio": calmar
    }

# Execute Backtests across all 6 strategies
spy_ret = df['SPY'].pct_change().fillna(0)
bench_ret = run_60_40_benchmark(df)
mrs_bin_ret = run_backtest_simulation(df, df['MRS_Prob'], 'binary')
mrs_con_ret = run_backtest_simulation(df, df['MRS_Prob'], 'continuous')
ml_bin_ret = run_backtest_simulation(df, df['ML_Prob'], 'binary')
ml_con_ret = run_backtest_simulation(df, df['ML_Prob'], 'continuous')

all_ret = pd.DataFrame({
    'S&P 500 Buy & Hold': spy_ret,
    'Benchmark (60/40)': bench_ret,
    'Markov Switching (MRS - Binary)': mrs_bin_ret,
    'Markov Switching (MRS - Continuous)': mrs_con_ret,
    'Machine Learning (RF - Binary)': ml_bin_ret,
    'Machine Learning (RF - Continuous)': ml_con_ret
}, index=df.index)

# Out-of-sample window (2010-01 to 2026-09)
eval_start = df.index[120]
eval_returns = all_ret[all_ret.index >= eval_start]
eval_df = df[df.index >= eval_start]
print(f"Backtesting complete for Out-of-Sample window: {eval_start.strftime('%Y-%m')} to {eval_df.index[-1].strftime('%Y-%m')}")


## 5. Master Performance Summary Table (All 6 Strategies)

In [ ]:
metrics_list = []
for col in eval_returns.columns:
    m = calculate_all_metrics(eval_returns[col])
    m['Strategy'] = col
    metrics_list.append(m)

master_table = pd.DataFrame(metrics_list).set_index('Strategy')

# Format table for presentation
formatted_table = master_table.copy()
formatted_table['Cumulative Return'] = (formatted_table['Cumulative Return'] * 100).round(2).astype(str) + '%'
formatted_table['Annualized Return'] = (formatted_table['Annualized Return'] * 100).round(2).astype(str) + '%'
formatted_table['Annualized Vol'] = (formatted_table['Annualized Vol'] * 100).round(2).astype(str) + '%'
formatted_table['Sharpe Ratio'] = formatted_table['Sharpe Ratio'].round(3)
formatted_table['Sortino Ratio'] = formatted_table['Sortino Ratio'].round(3)
formatted_table['Max Drawdown'] = (formatted_table['Max Drawdown'] * 100).round(2).astype(str) + '%'
formatted_table['Calmar Ratio'] = formatted_table['Calmar Ratio'].round(3)

display(formatted_table)


## 6. Visualizations & Analytical Charts

In [ ]:
# 1. Master Cumulative Returns Chart
cum_returns = (1 + eval_returns).cumprod() - 1

fig, ax = plt.subplots(figsize=(13, 7))
styles = {
    'S&P 500 Buy & Hold': {'color': '#212529', 'linestyle': '-', 'linewidth': 2.2},
    'Benchmark (60/40)': {'color': '#6c757d', 'linestyle': '--', 'linewidth': 1.8},
    'Markov Switching (MRS - Binary)': {'color': '#fd7e14', 'linestyle': ':', 'linewidth': 1.6},
    'Markov Switching (MRS - Continuous)': {'color': '#e83e8c', 'linestyle': '-.', 'linewidth': 1.6},
    'Machine Learning (RF - Binary)': {'color': '#198754', 'linestyle': '-', 'linewidth': 2.3},
    'Machine Learning (RF - Continuous)': {'color': '#0d6efd', 'linestyle': '-', 'linewidth': 1.8}
}

for col in cum_returns.columns:
    cfg = styles.get(col, {'color': 'blue', 'linestyle': '-', 'linewidth': 1.5})
    pct_series = cum_returns[col] * 100
    ax.plot(cum_returns.index, pct_series, label=f"{col} ({pct_series.iloc[-1]:.1f}%)", **cfg)
    
ax.axvspan(pd.to_datetime('2020-02-01'), pd.to_datetime('2020-04-01'), color='#fd7e14', alpha=0.15, label='2020 COVID Shock')
ax.axvspan(pd.to_datetime('2022-01-01'), pd.to_datetime('2022-10-01'), color='#ffc107', alpha=0.15, label='2022 Inflation & Rate Hike Cycle')
ax.set_title("Master Out-of-Sample Cumulative Returns Comparison (2010 - 2026)", fontsize=13, fontweight='bold', pad=12)
ax.set_ylabel("Cumulative Total Return (%)", fontsize=11)
ax.set_xlabel("Date", fontsize=11)
ax.grid(True, linestyle='--', alpha=0.5)
ax.legend(loc='upper left', frameon=True, fontsize=9.5)
plt.tight_layout()
plt.show()

# 2. Master Underwater Drawdown Comparison
drawdowns = pd.DataFrame(index=eval_returns.index)
for col in eval_returns.columns:
    w = (1 + eval_returns[col]).cumprod()
    drawdowns[col] = (w - w.cummax()) / w.cummax() * 100

fig_dd, ax_dd = plt.subplots(figsize=(13, 6))
for col in drawdowns.columns:
    cfg = styles.get(col, {'color': 'blue', 'linestyle': '-', 'linewidth': 1.5})
    ax_dd.plot(drawdowns.index, drawdowns[col], label=f"{col} (Max: {drawdowns[col].min():.2f}%)", **cfg)

ax_dd.axvspan(pd.to_datetime('2020-02-01'), pd.to_datetime('2020-04-01'), color='#fd7e14', alpha=0.15)
ax_dd.axvspan(pd.to_datetime('2022-01-01'), pd.to_datetime('2022-10-01'), color='#ffc107', alpha=0.15)
ax_dd.set_title("Master Out-of-Sample Underwater Drawdown Comparison (2010 - 2026)", fontsize=13, fontweight='bold', pad=12)
ax_dd.set_ylabel("Drawdown (%)", fontsize=11)
ax_dd.set_xlabel("Date", fontsize=11)
ax_dd.grid(True, linestyle='--', alpha=0.5)
ax_dd.legend(loc='lower left', frameon=True, fontsize=9.0, ncol=2)
plt.tight_layout()
plt.show()

# 3. Combined Regime Probability Trajectories
fig_reg, ax1 = plt.subplots(figsize=(13, 6))
ax1.plot(eval_df.index, eval_df['SPY'], color='#212529', linewidth=2.0, label='S&P 500 (SPY Price, Left Axis)')
ax1.set_ylabel("S&P 500 Price ($)", fontsize=11, fontweight='bold')
ax1.set_xlabel("Date", fontsize=11)
ax1.grid(True, linestyle='--', alpha=0.5)

ax2 = ax1.twinx()
ax2.plot(eval_df.index, eval_df['MRS_Prob'], color='#0d6efd', linewidth=1.5, alpha=0.85, label='Markov Risk-Off Prob (Blue)')
ax2.plot(eval_df.index, eval_df['ML_Prob'], color='#198754', linewidth=1.8, alpha=0.9, label='ML Risk-Off Prob (Green)')
ax2.axhline(0.50, color='red', linestyle='--', linewidth=1.2, label='50% Threshold')
ax2.fill_between(eval_df.index, 0, eval_df['MRS_Prob'], color='#0d6efd', alpha=0.1)
ax2.fill_between(eval_df.index, 0, eval_df['ML_Prob'], color='#198754', alpha=0.1)
ax2.set_ylabel("Regime Probability [0, 1]", fontsize=11, fontweight='bold')
ax2.set_ylim(-0.05, 1.05)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left', frameon=True, fontsize=9.5)
plt.title("S&P 500 Price & Combined Regime-Switching Probabilities (MRS vs ML)", fontsize=13, fontweight='bold', pad=12)
plt.tight_layout()
plt.show()

# 4. Gini Feature Importance Horizontal Bar Plot
mean_gini = np.mean(gini_importances_list, axis=0)
gini_df = pd.DataFrame({'Feature': features_ml, 'Importance': mean_gini * 100}).sort_values(by='Importance', ascending=True)

plt.figure(figsize=(10, 6))
bars = plt.barh(gini_df['Feature'], gini_df['Importance'], color='#198754', alpha=0.85, edgecolor='#0f5132')
for bar in bars:
    w = bar.get_width()
    plt.text(w + 0.2, bar.get_y() + bar.get_height()/2, f'{w:.2f}%', ha='left', va='center', fontsize=9.5, fontweight='bold')
plt.title('Random Forest Walk-Forward Mean Gini Feature Importance (2010 - 2026)', fontsize=12, fontweight='bold', pad=15)
plt.xlabel('Mean Gini Importance (Mean Decrease in Impurity, %)', fontsize=11)
plt.xlim(0, max(gini_df['Importance']) + 3)
plt.grid(axis='x', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()
